# Introduction to JAX and Neural Networks

JAX is a Python library from Google for writing functions that can be automatically differentiated, vectorized over data, and compiled for efficient execution. These are exactly the building blocks needed for training neural networks — and, as we'll see in the next notebook, for solving economic models.

We proceed in four steps:
1. **Automatic differentiation** — compute exact derivatives of Python functions
2. **Vectorization and compilation** — evaluate functions efficiently over batches of data
3. **Gradient descent** — use these tools to estimate an OLS regression
4. **Neural networks** — build a feedforward network and see it approximate arbitrary functions

## 1. Automatic Differentiation

JAX's `grad` function computes exact derivatives of Python functions via automatic differentiation. Unlike numerical differentiation (finite differences), this is exact and efficient.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap
import numpy as np
import matplotlib.pyplot as plt

jax.config.update('jax_platform_name', 'cpu')

Start with a simple function $f(x) = 7x^3 + x^2$. We can compute its derivative analytically: $f'(x) = 21x^2 + 2x$. Let's verify JAX gets the same answer.

In [ ]:
def f(x):
    return 7*x**3 + x**2

# JAX computes exact derivatives
print(f"f(2.0)   = {f(2.0)}")
print(f"f'(2.0)  = {grad(f)(2.0)}")      # should be 21*4 + 4 = 88
print(f"f''(2.0) = {grad(grad(f))(2.0)}") # should be 42*2 + 2 = 86

In [ ]:
def h(x):
    return jnp.sin(x) * jnp.cos(x) + f(jnp.sqrt(x)) / jnp.exp(x)

print(f"h'(2.0)  = {grad(h)(2.0)}")
print(f"h''(2.0) = {grad(grad(h))(2.0)}")

**Gradients of multivariate functions.** `grad` computes the gradient with respect to the first argument by default. For a function $f(\mathbf{x}) = (x_0 + \alpha x_1)^2$, the gradient is $\nabla_{\mathbf{x}} f = 2(x_0 + \alpha x_1) \cdot [1, \alpha]^T$.

In [ ]:
def g(x, alpha):
    return (x[0] + alpha * x[1])**2

grad_g = grad(g)  # gradient w.r.t. first argument (x)
print(grad_g(jnp.array([2.0, 3.0]), 3.0))  # should be 2*(2+9)*[1,3] = [22, 66]

## 2. Vectorization and Compilation

### vmap: automatic vectorization

`vmap` transforms a function written to process a **single** data point into one that handles an entire batch. Rather than writing an explicit loop over observations, you tell JAX which arguments vary across the batch and which stay fixed. In the call `vmap(f, in_axes=(None, 0, 0))`, `None` means "this argument is the same for every observation" (e.g. the parameter vector), while `0` means "iterate over the first dimension of this array" — i.e. over rows, so each observation gets its own row of `X` and element of `y`.

In [ ]:
def squared_error(params, y, x):
    """Squared error for a single observation."""
    pred = x @ params
    return (y - pred)**2

# vmap: vectorize over observations (y and x), keeping params fixed
batch_squared_error = vmap(squared_error, in_axes=(None, 0, 0))

To test this, we generate a small synthetic regression dataset: 5 observations with 3 features, where `y` is a known linear function of `X` plus a small amount of noise. Since we set `params` to the true coefficients, the squared errors should be close to zero.

In [ ]:
# Synthetic data: y = X @ [1, 2, 0.5] + small noise
key = jax.random.PRNGKey(42)
true_params = jnp.array([1.0, 2.0, 0.5])
X = jax.random.normal(key, shape=(5, 3))
y = X @ true_params + 0.1 * jax.random.normal(key, shape=(5,))

# Evaluate at the true parameters — errors should be near zero
errors = batch_squared_error(true_params, y, X)
print(f"Squared errors: {errors}")
print(f"MSE: {jnp.mean(errors):.6f}")

### jit: just-in-time compilation

Every time JAX executes a function, it traces through the Python code and dispatches operations one by one. `jit` short-circuits this: on the first call, it compiles the entire function into optimized machine code via [XLA](https://openxla.org/xla). Subsequent calls skip the Python overhead entirely and run the compiled version directly.

This matters most for functions called repeatedly in a loop — exactly the pattern in gradient descent training. Below we define a loss function using the `batch_squared_error` from above, wrap it with `value_and_grad` to get both the loss and its gradient, and then time 100 evaluations with and without `jit`.

In [ ]:
import time

def loss(params, y, X):
    return jnp.mean(batch_squared_error(params, y, X))

loss_and_grad = jax.value_and_grad(loss)
loss_and_grad_jit = jit(jax.value_and_grad(loss))

# Generate larger dataset for timing
key = jax.random.PRNGKey(0)
n, p = 10000, 20
X_large = jax.random.normal(key, shape=(n, p))
true_beta = jax.random.normal(key, shape=(p,))
y_large = X_large @ true_beta + 0.5 * jax.random.normal(key, shape=(n,))
params_init = jnp.zeros(p)

# Warm up JIT
_ = loss_and_grad_jit(params_init, y_large, X_large)

# Time without JIT
start = time.time()
for _ in range(100):
    loss_and_grad(params_init, y_large, X_large)
t_no_jit = time.time() - start

# Time with JIT
start = time.time()
for _ in range(100):
    loss_and_grad_jit(params_init, y_large, X_large)
t_jit = time.time() - start

print(f"Without JIT: {t_no_jit:.3f}s")
print(f"With JIT:    {t_jit:.3f}s")
print(f"Speedup:     {t_no_jit/t_jit:.1f}x")

## 3. Gradient descent for OLS

We now combine `grad`, `vmap`, and `jit` to estimate an OLS regression via gradient descent. The loss function is the mean squared error:
$$L(\boldsymbol{\beta}) = \frac{1}{N} \sum_{i=1}^N (y_i - \mathbf{x}_i^T \boldsymbol{\beta})^2$$
Rather than using the closed-form solution $\hat{\boldsymbol{\beta}} = (X^TX)^{-1}X^Ty$, we iteratively update $\boldsymbol{\beta}$ by moving in the direction of steepest descent: $\boldsymbol{\beta} \leftarrow \boldsymbol{\beta} - \eta \nabla_{\boldsymbol{\beta}} L$.

In [ ]:
# Generate synthetic data: y = X*beta + noise, noise ~ N(0, 8^2)
key = jax.random.PRNGKey(201)
n_obs, n_features = 1000, 5
true_betas = jnp.array([4.0, 3.0, 2.0, 6.0, 0.0])
noise_std = 8.0

key1, key2 = jax.random.split(key)
X = jax.random.normal(key1, shape=(n_obs, n_features)) * 4
y = X @ true_betas + jax.random.normal(key2, shape=(n_obs,)) * noise_std

print(f"Data: {n_obs} observations, {n_features} features")
print(f"True betas: {true_betas}")
print(f"Noise std:  {noise_std}")

In [ ]:
@jit
def loss_fn(params, y, X):
    preds = X @ params
    return jnp.mean((y - preds)**2)

loss_and_grad_fn = jit(jax.value_and_grad(loss_fn))

def gradient_descent(params, y, X, lr=0.001, epochs=2000):
    losses = []
    for epoch in range(epochs):
        loss_val, grads = loss_and_grad_fn(params, y, X)
        params = params - lr * grads
        losses.append(loss_val)
        if epoch % 500 == 0:
            print(f"Epoch {epoch:4d}: loss = {loss_val:.4f}")
    return params, losses

params_init = jax.random.normal(jax.random.PRNGKey(0), shape=(n_features,))
params_gd, losses = gradient_descent(params_init, y, X)

In [ ]:
# Closed-form OLS
beta_ols = jnp.linalg.solve(X.T @ X, X.T @ y)

print(f"{'':12s} {'True':>8s} {'GD':>8s} {'OLS':>8s}")
print("-" * 40)
for j in range(n_features):
    print(f"  beta_{j}:    {true_betas[j]:8.3f} {params_gd[j]:8.3f} {beta_ols[j]:8.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.set_title("Gradient descent convergence")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 4. Neural networks as universal approximators

A feedforward neural network with even a single hidden layer can approximate any continuous function to arbitrary precision (Cybenko, 1989; Hornik et al., 1989). We now build a simple one-hidden-layer network from scratch using the JAX primitives introduced above.

A single-hidden-layer network with $H$ hidden units computes:
$$\hat{y} = \mathbf{W}_2 \, \sigma(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + b_2$$
where $\sigma$ is a nonlinear activation function (here sigmoid), and $\theta = \{\mathbf{W}_1, \mathbf{b}_1, \mathbf{W}_2, b_2\}$ are the parameters learned by gradient descent.

In [ ]:
@jit
def forward(params, X):
    """Forward pass: one hidden layer with sigmoid activation."""
    h = jax.nn.sigmoid(X @ params["W1"] + params["b1"])
    return h @ params["W2"] + params["b2"]

@jit
def nn_loss(params, X, y):
    preds = forward(params, X)
    return jnp.mean((y - preds)**2)

nn_loss_and_grad = jit(jax.value_and_grad(nn_loss))

def init_params(key, input_dim, num_units):
    keys = jax.random.split(key, 4)
    return {
        "W1": jax.random.uniform(keys[0], (input_dim, num_units), minval=-1, maxval=1),
        "b1": jax.random.uniform(keys[1], (num_units,), minval=-1, maxval=1),
        "W2": jax.random.uniform(keys[2], (num_units, 1), minval=-1, maxval=1),
        "b2": jax.random.uniform(keys[3], (1,), minval=-1, maxval=1),
    }

def train_nn(key, X, y, num_units=20, lr=0.01, epochs=10000):
    # The forward pass expects X as a 2D matrix (observations x features).
    # For scalar inputs like linspace output, reshape from (N,) to (N, 1).
    X_2d = X.reshape(-1, 1) if X.ndim == 1 else X
    params = init_params(key, X_2d.shape[1], num_units)
    losses = []
    for epoch in range(epochs):
        loss_val, grads = nn_loss_and_grad(params, X_2d, y.reshape(-1, 1))
        params = jax.tree.map(lambda p, g: p - lr * g, params, grads)
        losses.append(loss_val)
    return params, losses

### Test functions

We define several functions of varying complexity to test the network's approximation ability.

In [ ]:
x = np.linspace(-5, 5, 100)

functions = {
    "Step": np.where(x < 0, -1.0, 1.0),
    "Absolute value": np.abs(x),
    "Cosine": np.cos(x),
    "Cubic": x**3 / 50,  # scaled for easier training
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, y_fn) in zip(axes, functions.items()):
    ax.plot(x, y_fn, "k-", linewidth=2)
    ax.set_title(name)
plt.tight_layout()
plt.show()

### Approximation with 3 hidden units

With only 3 hidden units, the network can capture broad shape but struggles with sharp features.

In [ ]:
key = jax.random.PRNGKey(92)
x_jax = jnp.array(x)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (name, y_fn) in zip(axes, functions.items()):
    y_jax = jnp.array(y_fn)
    params, losses = train_nn(key, x_jax, y_jax, num_units=3, lr=0.005, epochs=10000)
    y_pred = forward(params, x_jax.reshape(-1, 1))
    
    ax.plot(x, y_fn, "k-", linewidth=2, label="True")
    ax.plot(x, np.array(y_pred), "C1-", linewidth=2, label="NN (3 units)")
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle("Neural network approximation: 3 hidden units", y=1.02)
plt.tight_layout()
plt.show()

### Approximation with 20 hidden units

Increasing to 20 hidden units dramatically improves the fit — the network can now capture sharp transitions and fine detail.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (name, y_fn) in zip(axes, functions.items()):
    y_jax = jnp.array(y_fn)
    params, losses = train_nn(key, x_jax, y_jax, num_units=20, lr=0.005, epochs=10000)
    y_pred = forward(params, x_jax.reshape(-1, 1))
    
    ax.plot(x, y_fn, "k-", linewidth=2, label="True")
    ax.plot(x, np.array(y_pred), "C2-", linewidth=2, label="NN (20 units)")
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle("Neural network approximation: 20 hidden units", y=1.02)
plt.tight_layout()
plt.show()

### What's next?

We've seen that JAX provides three key primitives — `grad`, `vmap`, `jit` — and that neural networks built from these primitives can approximate arbitrary functions. In the next notebook, we apply these ideas to solve the neoclassical growth model: instead of fitting data, we train a neural network to satisfy the Hamilton-Jacobi-Bellman equation.